# Test 1·2 추가 실험

$\varepsilon=10^{-4}$ 고정. `p order (기존)`은 $M=1024$, $N_{\rm ref}=2048$에서 원래 차분식으로 계산한 관측률이다.

In [4]:
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <vector>
#include "../include/BlackScholesSolver.h"
#include "../include/ErrorAnalysis.h"

const int count = 4;
int nValues[count] = {128, 256, 512, 1024};
const int originalM = 1024;
const int Nref = 2048;
const double epsilon = 1e-4;
double originalErrors[2][count];
double coupledErrors[2][count];
double junctionErrors[2][count];

In [ ]:
for(int test=1; test<=2; ++test)
{
    const double alpha = test == 1 ? 0.04 : 0.16;
    BlackScholesSolver reference(Nref, originalM, alpha, 0.06, epsilon);
    if(test == 1) reference.solveTest1(); else reference.solveTest2();
    for(int k=0; k<count; ++k)
    {
        BlackScholesSolver coarse(nValues[k], originalM, alpha, 0.06, epsilon);
        if(test == 1) coarse.solveTest1(); else coarse.solveTest2();
        originalErrors[test-1][k] = ErrorAnalysis::maxError(coarse, reference);
    }
}

void printOrders(int test, const int* mValues, const double* errors)
{
    printf("Test %d\n", test);
    printf("     M     N       computed e    p order    p order (기존)\n");
    for(int k=0; k<count; ++k)
    {
        printf("%6d %5d   %14.8e  ", mValues[k], nValues[k], errors[k]);
        if(k+1<count)
        {
            double order = log(errors[k]/errors[k+1])/log(2.0);
            double originalOrder = log(originalErrors[test-1][k]/originalErrors[test-1][k+1])/log(2.0);
            printf("%9.3f %17.3f\n", order, originalOrder);
        }
        else printf("%9s %17s\n", "-", "-");
    }
    printf("\n");
}

## 추가 실험 1 · $\tau\propto h^2$

$M=N^2/16$으로 두고 $(N,M)$과 $(2N,4M)$의 전체 시공간 최대차이를 계산한다. coarse의 $k$번째 시간은 fine의 $4k$번째 시간과 일치한다.

In [6]:
struct TimeStepper
{
    BlackScholesSolver grid;
    int test, M;
    std::vector<double> expx;

    TimeStepper(int test_, int N, int M_)
        : grid(N, 1, test_ == 1 ? 0.04 : 0.16, 0.06, epsilon),
          test(test_), M(M_), expx(N+1)
    {
        grid.tau = 1.0/M;
        for(int i=0; i<=N; ++i)
        {
            grid.value(i, 1) = grid.payoff(grid.x[i]-grid.K);
            expx[i] = exp(-grid.x[i]);
            if(test == 2)
            {
                double sigma = 0.4*(2.0+sin(grid.x[i]));
                grid.sigmaSquared[i] = sigma*sigma;
            }
        }
        if(test == 2) grid.assembleCoefficients(0);
    }

    void advance(int k)
    {
        double s = double(k)/M;
        if(test == 1)
        {
            for(int i=0; i<=grid.N; ++i)
            {
                double sigma = 0.2*(1.0+(1.0-s)*expx[i]);
                grid.sigmaSquared[i] = sigma*sigma;
            }
            grid.assembleCoefficients(0);
        }
        grid.value(0, 0) = 0.0;
        grid.value(grid.N, 0) = grid.Smax-grid.K*exp(-grid.r*s);
        grid.step(0);
        std::copy(grid.U, grid.U+grid.N+1, grid.U+grid.N+1);
    }
};

int coupledM[count];
for(int k=0; k<count; ++k) coupledM[k] = nValues[k]*nValues[k]/16;

for(int test=1; test<=2; ++test)
{
    for(int row=0; row<count; ++row)
    {
        int N = nValues[row], M = coupledM[row];
        TimeStepper coarse(test, N, M), fine(test, 2*N, 4*M);
        std::vector<int> left(N+1);
        std::vector<double> weights(N+1);
        int j = 0;
        for(int i=0; i<=N; ++i)
        {
            while(j<fine.grid.N-1 && fine.grid.x[j+1]<coarse.grid.x[i]) ++j;
            left[i] = j;
            weights[i] = (coarse.grid.x[i]-fine.grid.x[j])/fine.grid.h[j+1];
        }

        double error = 0.0;
        for(int k=0; k<=M; ++k)
        {
            if(k>0)
            {
                coarse.advance(k);
                for(int q=4*k-3; q<=4*k; ++q) fine.advance(q);
            }
            for(int i=0; i<=N; ++i)
            {
                int p = left[i];
                double w = weights[i];
                double interpolated = (1.0-w)*fine.grid.value(p, 1)+w*fine.grid.value(p+1, 1);
                error = std::max(error, fabs(coarse.grid.value(i, 1)-interpolated));
            }
        }
        coupledErrors[test-1][row] = error;
    }
    printOrders(test, coupledM, coupledErrors[test-1]);
}

Test 1
     M     N       computed e    p order    p order (기존)
  1024   128   5.03908626e-02      1.014             1.122
  4096   256   2.49547653e-02      1.008             1.312
 16384   512   1.24109636e-02      1.006             1.880
 65536  1024   6.18163393e-03          -                 -

Test 2
     M     N       computed e    p order    p order (기존)
  1024   128   6.03885518e-02      1.014             1.551
  4096   256   2.98979452e-02      1.009             1.919
 16384   512   1.48556160e-02      1.006             2.284
 65536  1024   7.39445849e-03          -                 -



## 추가 실험 2 · 접합부 미분식 교체

같은 격자에서 $D_x$만 비균일 3점식으로 교체한다. $M=1024$, $N_{\rm ref}=2048$이다.

In [7]:
void solveThreePoint(BlackScholesSolver& grid, int test)
{
    grid.setTerminalPayoff();
    if(test == 2)
        for(int i=0; i<=grid.N; ++i)
        {
            double sigma = 0.4*(2.0+sin(grid.x[i]));
            grid.sigmaSquared[i] = sigma*sigma;
        }

    for(int j=grid.M-1; j>=0; --j)
    {
        if(test == 1)
            for(int i=0; i<=grid.N; ++i)
            {
                double sigma = 0.2*(1.0+j*grid.tau*exp(-grid.x[i]));
                grid.sigmaSquared[i] = sigma*sigma;
            }

        for(int i=1; i<grid.N; ++i)
        {
            double l = grid.h[i], d = grid.h[i+1];
            double twoA = grid.sigmaSquared[i]*grid.x[i]*grid.x[i];
            double b = grid.r*grid.x[i];
            grid.A[i] = (-twoA+b*d)/(l*(l+d));
            grid.B[i] = 1.0/grid.tau+(twoA-b*(d-l))/(l*d)+grid.r;
            grid.C[i] = (-twoA-b*l)/(d*(l+d));
        }
        grid.step(j);
    }
}

int junctionM[count];
for(int k=0; k<count; ++k) junctionM[k] = originalM;

for(int test=1; test<=2; ++test)
{
    double alpha = test == 1 ? 0.04 : 0.16;
    BlackScholesSolver reference(Nref, originalM, alpha, 0.06, epsilon);
    solveThreePoint(reference, test);
    for(int k=0; k<count; ++k)
    {
        BlackScholesSolver coarse(nValues[k], originalM, alpha, 0.06, epsilon);
        solveThreePoint(coarse, test);
        junctionErrors[test-1][k] = ErrorAnalysis::maxError(coarse, reference);
    }
    printOrders(test, junctionM, junctionErrors[test-1]);
}

Test 1
     M     N       computed e    p order    p order (기존)
  1024   128   6.80058538e-02      1.122             1.122
  1024   256   3.12552951e-02      1.312             1.312
  1024   512   1.25872969e-02      1.880             1.880
  1024  1024   3.41877033e-03          -                 -

Test 2
     M     N       computed e    p order    p order (기존)
  1024   128   5.71090114e-02      1.551             1.551
  1024   256   1.94953921e-02      1.919             1.919
  1024   512   5.15459605e-03      2.284             2.284
  1024  1024   1.05858758e-03          -                 -

